In [ ]:
# ─────────────────────────────────────────────
# Install Required Libraries (run this first)
# ─────────────────────────────────────────────
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn --quiet
print('All libraries installed successfully.')

# Customer Segmentation for Targeted Marketing
## Business Intelligence and Data Mining — UFCFMM
### Group Project Portfolio

---

| Role | Member | Responsibility |
|------|--------|----------------|
| Problem Researcher | Ubah | Stage 1: Problem Definition & Data Scoping |
| Data Engineer | Ahmed | Stage 2: EDA & Pre-processing |
| ML Engineer | Amjad | Stage 3: Data Mining & Business Insights |
| Evaluator | [4th Member] | Stage 4: Evaluation, Ethics & Recommendations |

---

---
# Stage 1: Problem Definition and Data Scoping
**Author: Ubah**

---

## 1.1 Business Problem

In the modern retail environment, organisations face increasing pressure to maximise the return on their marketing expenditure. A fundamental challenge is that customers are not homogeneous — they differ significantly in their purchasing behaviour, income levels, and responsiveness to promotional activity. Treating all customers identically results in wasted marketing spend, poor customer experience, and missed revenue opportunities (Kotler and Keller, 2016).

This project addresses the following core business question:

> **How can a retail organisation group its customers based on purchasing behaviour and demographic characteristics in order to design more targeted, effective marketing campaigns?**

Customer segmentation — the process of dividing a customer base into distinct groups that share similar characteristics — is a well-established business intelligence technique that enables organisations to:

- **Personalise marketing messages** for each customer segment, increasing campaign relevance and conversion rates.
- **Allocate marketing budgets more efficiently** by prioritising high-value customer groups.
- **Improve customer retention** by identifying at-risk or under-served segments and tailoring retention strategies accordingly.
- **Identify new product opportunities** by understanding the unmet needs of specific groups.

According to McKinsey & Company (2021), personalisation at scale can deliver five to eight times the ROI on marketing spend and lift sales by 10% or more. Similarly, Wedel and Kamakura (2000) argue that effective market segmentation is a prerequisite for competitive differentiation in saturated retail markets.

This project will apply unsupervised machine learning — specifically K-Means clustering — to a real-world retail dataset to identify meaningful customer segments and translate these into actionable marketing strategies.

## 1.2 Dataset Selection and Justification

### Dataset: Mall Customer Segmentation Dataset
**Source:** Kaggle (https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)  
**Format:** CSV  
**Records:** 200 customers  
**Attributes:** 5

| Attribute | Type | Description |
|-----------|------|-------------|
| CustomerID | Integer | Unique customer identifier |
| Gender | Categorical | Male / Female |
| Age | Integer | Customer age in years |
| Annual Income (k$) | Integer | Annual income in thousands of USD |
| Spending Score (1–100) | Integer | Score assigned by the mall based on spending behaviour and loyalty |

### Why This Dataset is Appropriate

This dataset is well-suited to the stated business problem for several reasons:

1. **Direct relevance:** The attributes — particularly Annual Income and Spending Score — directly capture the economic and behavioural dimensions required for meaningful customer segmentation.
2. **Cleanliness and completeness:** The dataset contains no missing values and is well-documented, allowing the team to focus analytical effort on insight generation rather than extensive data repair.
3. **Appropriate size for exploration:** At 200 records, the dataset is large enough to identify meaningful clusters while remaining computationally accessible and interpretable.
4. **Wide academic and industry use:** This dataset has been widely used in published tutorials and academic exercises (Chaudhary, 2019), providing a strong basis for benchmarking and comparison.

### Limitations

It is important to acknowledge the following limitations:

- **Small sample size:** 200 records limits the statistical robustness of findings and may not fully represent the diversity of a real customer base.
- **Simplified attributes:** Real-world segmentation typically incorporates purchase history, product category preferences, and recency/frequency/monetary (RFM) data. This dataset lacks such granularity.
- **Synthetic origin:** The Spending Score is an assigned metric rather than a directly observed behaviour, which may introduce bias in how purchasing intent is represented.
- **Temporal dimension absent:** There is no timestamp data, so the analysis cannot account for changes in customer behaviour over time.

Despite these limitations, the dataset provides a solid and pedagogically appropriate foundation for demonstrating the end-to-end business intelligence pipeline.

### References

Chaudhary, V. (2019) *Customer Segmentation Tutorial in Python*. Kaggle. Available at: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python (Accessed: 10 March 2026).

Kotler, P. and Keller, K.L. (2016) *Marketing Management*. 15th edn. Harlow: Pearson Education.

McKinsey & Company (2021) *The value of getting personalisation right — or wrong — is multiplying*. Available at: https://www.mckinsey.com/capabilities/growth-marketing-and-sales/our-insights/the-value-of-getting-personalization-right-or-wrong-is-multiplying (Accessed: 10 March 2026).

Wedel, M. and Kamakura, W.A. (2000) *Market Segmentation: Conceptual and Methodological Foundations*. 2nd edn. Boston: Kluwer Academic Publishers.

---
# Stage 2: Exploratory Data Analysis (EDA) and Pre-processing
**Author: Ahmed**

---

In [ ]:
# ─────────────────────────────────────────────
# Import Libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Libraries loaded successfully.')

In [ ]:
# ─────────────────────────────────────────────
# Load Dataset
# ─────────────────────────────────────────────
df = pd.read_csv('Mall_Customers.csv')

# Rename columns for convenience
df.columns = ['CustomerID', 'Gender', 'Age', 'Annual_Income', 'Spending_Score']

print(f'Dataset shape: {df.shape}')
df.head(10)

**Observation:** The dataset contains 200 rows and 5 columns. Each row represents a single customer. The columns align with the data dictionary described in Stage 1. We have renamed columns to remove spaces, which simplifies downstream referencing in code.

In [ ]:
# ─────────────────────────────────────────────
# Data Types and Structure
# ─────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Statistical Summary ===')
df.describe()

**Observation:** All numeric columns are integer types, which is appropriate. The statistical summary reveals:
- **Age** ranges from 18 to 70, with a mean of approximately 38.9 years, suggesting a broad adult customer base.
- **Annual Income** ranges from 15k to 137k USD, with a mean of approximately 60.6k, indicating considerable income diversity.
- **Spending Score** ranges from 1 to 99, with a mean of approximately 50.2, suggesting the scores are roughly uniformly distributed across the range.

This diversity across all three numeric attributes is encouraging — it suggests the data has sufficient variance to support meaningful clustering.

In [ ]:
# ─────────────────────────────────────────────
# Missing Values Check
# ─────────────────────────────────────────────
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print(f'Total missing values: {df.isnull().sum().sum()}')
print()
print('=== Duplicate Rows ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

**Observation:** The dataset contains no missing values and no duplicate rows. This is a significant advantage — no imputation or deduplication is required, and the dataset can proceed directly to transformation and analysis. This finding is consistent with the dataset's documented nature as a clean, curated resource.

In [ ]:
# ─────────────────────────────────────────────
# Gender Distribution
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gender_counts = df['Gender'].value_counts()

# Bar chart
axes[0].bar(gender_counts.index, gender_counts.values, color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Gender Distribution (Count)')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(gender_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Gender Distribution (%)')

plt.suptitle('Customer Gender Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('gender_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The customer base is 56% female and 44% male — a moderate gender imbalance. This has implications for marketing: campaigns targeting the dataset as a whole will skew toward female customer preferences. Gender-disaggregated analysis in Stage 3 will help determine whether spending behaviour differs meaningfully by gender, which could inform whether gender-specific campaign variants are warranted.

In [ ]:
# ─────────────────────────────────────────────
# Distribution Plots: Age, Income, Spending Score
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

features = ['Age', 'Annual_Income', 'Spending_Score']
colors   = ['#4C72B0', '#55A868', '#C44E52']
labels   = ['Age (years)', 'Annual Income (k$)', 'Spending Score (1–100)']

for ax, feat, col, lab in zip(axes, features, colors, labels):
    ax.hist(df[feat], bins=20, color=col, edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.axvline(df[feat].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {df[feat].mean():.1f}')
    ax.set_title(f'Distribution of {lab}')
    ax.set_xlabel(lab)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations:**

- **Age:** The distribution is right-skewed, with a concentration of customers in the 25–45 age bracket. Fewer older customers (60+) are present. This is typical of a shopping mall demographic.
- **Annual Income:** Approximately normally distributed with a slight right skew, peaking around 60–70k. The presence of high-income customers (100k+) suggests a premium segment exists.
- **Spending Score:** Remarkably uniform across the 1–100 range, with slight peaks at both extremes. This bimodal tendency hints at distinct groups of high and low spenders — an early indication that clustering may reveal meaningful sub-groups.

In [ ]:
# ─────────────────────────────────────────────
# Box Plots: Outlier Detection
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat, col, lab in zip(axes, features, colors, labels):
    ax.boxplot(df[feat], patch_artist=True,
               boxprops=dict(facecolor=col, alpha=0.7),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(f'Box Plot: {lab}')
    ax.set_ylabel(lab)
    ax.set_xticks([])

plt.suptitle('Outlier Detection via Box Plots', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The box plots confirm that no extreme outliers are present in any of the three numeric features. While there are a small number of high-income data points beyond the upper whisker, these represent genuine high-income customers rather than data errors and should be retained. No data removal is necessary at this stage, which preserves the full 200-record dataset for modelling.

In [ ]:
# ─────────────────────────────────────────────
# Correlation Heatmap
# ─────────────────────────────────────────────
plt.figure(figsize=(7, 5))

corr = df[['Age', 'Annual_Income', 'Spending_Score']].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 13, 'weight': 'bold'})

plt.title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The correlation matrix reveals weak linear relationships between all three numeric features. Notably:
- The correlation between **Age and Spending Score is weakly negative (≈ −0.33)**, suggesting older customers tend to spend slightly less — though this is not a strong relationship.
- **Annual Income and Spending Score are nearly uncorrelated (≈ 0.01)**, which is a key insight: earning more does not automatically mean spending more in this dataset. This non-linear relationship is exactly why clustering (rather than regression) is the appropriate technique — it can identify distinct behavioural groups that a linear model would miss.

In [ ]:
# ─────────────────────────────────────────────
# Pairplot
# ─────────────────────────────────────────────
pair = sns.pairplot(df[['Age', 'Annual_Income', 'Spending_Score', 'Gender']],
                    hue='Gender', palette={'Male': '#4C72B0', 'Female': '#DD8452'},
                    diag_kind='kde', plot_kws={'alpha': 0.6})
pair.fig.suptitle('Pairplot of Numeric Features by Gender', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The pairplot provides the most informative single view of the dataset. The **Annual Income vs Spending Score** scatter plot is particularly revealing: a clear visual pattern of five distinct clusters is already visible — groups of customers that differ in both income and spending behaviour. This strongly motivates the use of K-Means clustering in Stage 3. Gender does not appear to be the primary driver of clustering, as male and female points are interleaved across all groups.

In [ ]:
# ─────────────────────────────────────────────
# Encode Categorical Variable
# ─────────────────────────────────────────────
df['Gender_Encoded'] = df['Gender'].map({'Male': 0, 'Female': 1})

print('Gender encoding applied:')
print(df[['Gender', 'Gender_Encoded']].drop_duplicates())

**Pre-processing Summary:**

The following pre-processing steps were applied:

1. **Column renaming** — whitespace removed from column headers for cleaner code.
2. **Missing value check** — confirmed zero missing values; no imputation required.
3. **Duplicate check** — confirmed zero duplicates; no removal required.
4. **Outlier assessment** — no harmful outliers identified; all 200 records retained.
5. **Categorical encoding** — Gender encoded as binary (Male=0, Female=1) for potential use in multi-dimensional analysis.

The dataset is now clean and ready for the data mining stage. Feature scaling will be applied in Stage 3 immediately prior to clustering, as K-Means is sensitive to differences in feature magnitude.

---
# Stage 3: Data Mining and Business Insights
**Author: Amjad**

---

## 3.1 Technique Selection and Justification

Given the business objective — grouping customers into meaningful segments without pre-defined labels — this is an **unsupervised learning** problem. No labelled outcome variable exists (e.g. there is no column indicating which segment a customer belongs to), which rules out supervised classification techniques.

**K-Means clustering** was selected as the primary technique for the following reasons:

1. **Appropriateness for segmentation:** K-Means is one of the most widely applied algorithms for customer segmentation in both academic literature and industry practice (Han, Kamber and Pei, 2012). It partitions data into K groups by minimising intra-cluster variance, which aligns well with the goal of finding groups of similar customers.

2. **Interpretability:** K-Means produces hard cluster assignments and easily interpretable cluster centroids, which can be directly translated into actionable business profiles.

3. **Computational efficiency:** For a dataset of 200 records, K-Means is computationally trivial and allows rapid iteration on different values of K.

**Alternatives considered:**
- *DBSCAN*: Capable of identifying arbitrarily shaped clusters and handling noise, but less suited to this dataset where clusters appear roughly spherical (as observed in the pairplot), and it requires careful tuning of epsilon and min_samples.
- *Hierarchical Clustering*: Provides a dendrogram for visualising cluster hierarchy, but is computationally expensive at scale and does not outperform K-Means for this use case.

The primary clustering will use **Annual Income** and **Spending Score** as features, as these two attributes most directly capture purchasing behaviour — the core of the business question. A secondary analysis will incorporate Age as a third dimension.

---
**Reference:**  
Han, J., Kamber, M. and Pei, J. (2012) *Data Mining: Concepts and Techniques*. 3rd edn. Waltham: Morgan Kaufmann.

In [ ]:
# ─────────────────────────────────────────────
# Feature Selection and Scaling
# ─────────────────────────────────────────────
X = df[['Annual_Income', 'Spending_Score']].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature matrix shape:', X.shape)
print()
print('Before scaling (first 5 rows):')
print(X.head())
print()
print('After scaling (first 5 rows):')
print(pd.DataFrame(X_scaled, columns=['Annual_Income_Scaled', 'Spending_Score_Scaled']).head())

**Justification for Scaling:** K-Means uses Euclidean distance to assign data points to clusters. Without scaling, Annual Income (range: 15–137) would dominate Spending Score (range: 1–99) simply due to its larger magnitude, producing biased cluster assignments. StandardScaler transforms each feature to have a mean of 0 and standard deviation of 1, ensuring both features contribute equally to the distance calculation.

In [ ]:
# ─────────────────────────────────────────────
# Elbow Method — Optimal K
# ─────────────────────────────────────────────
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

# Plot
plt.figure(figsize=(9, 5))
plt.plot(K_range, inertia, marker='o', markersize=8, linewidth=2.5,
         color='#4C72B0', markerfacecolor='#C44E52', markeredgecolor='white', markeredgewidth=1.5)
plt.axvline(x=5, color='#C44E52', linestyle='--', linewidth=1.5, label='Suggested K = 5')
plt.title('Elbow Method for Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(K_range)
plt.legend()
plt.tight_layout()
plt.savefig('elbow.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The elbow plot shows a clear inflection point at **K = 5**, where the rate of inertia reduction begins to flatten significantly. Beyond K = 5, adding additional clusters yields diminishing returns in terms of variance explained. This strongly suggests that five is the optimal number of clusters for this dataset, which we will confirm using the Silhouette Score below.

In [ ]:
# ─────────────────────────────────────────────
# Silhouette Score Analysis
# ─────────────────────────────────────────────
sil_scores = []

for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f'K = {k:2d}  |  Silhouette Score = {score:.4f}')

# Plot
plt.figure(figsize=(9, 5))
plt.bar(range(2, 11), sil_scores, color='#55A868', edgecolor='white', linewidth=1.2)
plt.axhline(y=max(sil_scores), color='#C44E52', linestyle='--', linewidth=1.5,
            label=f'Best Score = {max(sil_scores):.4f} at K={sil_scores.index(max(sil_scores))+2}')
plt.title('Silhouette Score by Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(range(2, 11))
plt.legend()
plt.tight_layout()
plt.savefig('silhouette_bar.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The Silhouette Score confirms the Elbow Method finding. **K = 5 achieves the highest silhouette score**, indicating that at this configuration, clusters are both cohesive (data points within a cluster are similar to each other) and well-separated (clusters are distinct from one another). A silhouette score above 0.5 is generally considered indicative of a reasonable cluster structure (Rousseeuw, 1987).

**Reference:**  
Rousseeuw, P.J. (1987) 'Silhouettes: A graphical aid to the interpretation and validation of cluster analysis', *Journal of Computational and Applied Mathematics*, 20, pp. 53–65.

In [ ]:
# ─────────────────────────────────────────────
# Final K-Means Model (K = 5)
# ─────────────────────────────────────────────
OPTIMAL_K = 5

kmeans = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

centroids_df = pd.DataFrame(centroids_original,
                             columns=['Annual_Income_Centroid', 'Spending_Score_Centroid'])
centroids_df.index.name = 'Cluster'

print(f'Final model fitted with K = {OPTIMAL_K}')
print(f'Inertia: {kmeans.inertia_:.2f}')
print(f'Final Silhouette Score: {silhouette_score(X_scaled, df["Cluster"]):.4f}')
print()
print('Cluster Centroids (original scale):')
print(centroids_df.round(1))

In [ ]:
# ─────────────────────────────────────────────
# Cluster Visualisation
# ─────────────────────────────────────────────
palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#9467BD']

plt.figure(figsize=(11, 7))

for cluster_id in range(OPTIMAL_K):
    cluster_data = df[df['Cluster'] == cluster_id]
    plt.scatter(cluster_data['Annual_Income'], cluster_data['Spending_Score'],
                c=palette[cluster_id], label=f'Cluster {cluster_id}',
                s=80, alpha=0.8, edgecolors='white', linewidths=0.8)

# Plot centroids
plt.scatter(centroids_original[:, 0], centroids_original[:, 1],
            c='black', s=250, marker='X', zorder=5, label='Centroids',
            edgecolors='white', linewidths=1.5)

plt.title('K-Means Customer Segmentation (K = 5)\nAnnual Income vs Spending Score',
          fontsize=14, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1–100)', fontsize=12)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Cluster Profile Summary
# ─────────────────────────────────────────────
profile = df.groupby('Cluster').agg(
    Count=('CustomerID', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income=('Annual_Income', 'mean'),
    Avg_Spending=('Spending_Score', 'mean'),
    Pct_Female=('Gender_Encoded', 'mean')
).round(1)

profile['Pct_Female'] = (profile['Pct_Female'] * 100).round(1).astype(str) + '%'

print('=== Cluster Profile Summary ===')
print(profile.to_string())

In [ ]:
# ─────────────────────────────────────────────
# Cluster Profiles — Visual Dashboard
# ─────────────────────────────────────────────
profile_num = df.groupby('Cluster').agg(
    Avg_Age=('Age', 'mean'),
    Avg_Income=('Annual_Income', 'mean'),
    Avg_Spending=('Spending_Score', 'mean'),
    Count=('CustomerID', 'count')
).round(1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['Avg_Income', 'Avg_Spending', 'Avg_Age']
ylabels = ['Avg Annual Income (k$)', 'Avg Spending Score', 'Avg Age (years)']
titles  = ['Average Income by Cluster', 'Average Spending Score by Cluster', 'Average Age by Cluster']

for ax, metric, ylabel, title in zip(axes, metrics, ylabels, titles):
    bars = ax.bar(profile_num.index, profile_num[metric],
                  color=palette, edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(OPTIMAL_K))
    for bar, val in zip(bars, profile_num[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Cluster Profile Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2 Business Interpretation of Clusters

The five clusters identified can be translated into distinct customer personas with specific strategic implications. The following profiles are derived from the cluster centroid values and demographic summaries.

> **Note:** Cluster IDs (0–4) are assigned by the algorithm and may vary across runs. The labels below are based on centroid characteristics observed in this execution.

---

### Cluster 0 — "The Careful Savers" 🟦
**Profile:** Mid-to-high income, very low spending score.  
**Interpretation:** These customers have the financial means to spend but choose not to. They may be deal-sensitive, brand-unaware, or unconvinced of value.  
**Business Recommendation:** Targeted value proposition campaigns — emphasise quality, exclusivity, or loyalty rewards. Consider personalised discount offers to convert this latent purchasing power into active spending.

---

### Cluster 1 — "The Premium Loyalists" 🟧
**Profile:** High income, high spending score.  
**Interpretation:** The most commercially valuable segment. These customers both earn well and spend freely. They are likely responsive to premium product lines, exclusive experiences, and VIP programmes.  
**Business Recommendation:** Invest in premium retention strategies — concierge-level customer service, early access to new products, invitation-only events. Prioritise this segment in high-margin campaign spend.

---

### Cluster 2 — "The Budget Conscious" 🟩
**Profile:** Low income, low spending score.  
**Interpretation:** These customers have constrained purchasing power and spend conservatively. They represent the largest segment by volume in many retail environments.  
**Business Recommendation:** Focus on value-for-money messaging, bundle deals, and discount promotions. Loyalty points programmes may encourage repeat visits without requiring large individual transactions.

---

### Cluster 3 — "The Impulsive Spenders" 🟥
**Profile:** Low income, high spending score.  
**Interpretation:** Despite modest incomes, these customers spend heavily — possibly driven by emotional purchasing, brand affinity, or social influences. They may be susceptible to overspending.  
**Business Recommendation:** Flash sales, limited-time offers, and social proof marketing (e.g., influencer campaigns) are likely to be effective. Buy-Now-Pay-Later options could also increase basket size, though ethical considerations apply (see Stage 4).

---

### Cluster 4 — "The Mainstream Middle" 🟪
**Profile:** Mid-range income, mid-range spending score.  
**Interpretation:** The average customer — neither extreme in income nor spending behaviour. This is the baseline customer that general marketing campaigns currently address.  
**Business Recommendation:** Standard seasonal campaigns, loyalty points, and cross-selling are appropriate. Personalisation opportunities exist by further sub-segmenting this group by age or product category preferences.

---

### Strategic Summary

| Cluster | Business Label | Avg Income | Avg Spending | Priority |
|---------|----------------|------------|--------------|----------|
| 1 | Premium Loyalists | High | High | ⭐ Highest |
| 3 | Impulsive Spenders | Low | High | ⭐ High |
| 0 | Careful Savers | High | Low | 🔼 Medium (growth opportunity) |
| 4 | Mainstream Middle | Mid | Mid | 🔼 Medium |
| 2 | Budget Conscious | Low | Low | 🔽 Lower (volume play) |

This segmentation enables the organisation to move from a one-size-fits-all marketing approach to a differentiated strategy that allocates budget in proportion to expected return — a fundamental principle of strategic marketing (Kotler and Keller, 2016).

---
# Stage 4: Evaluation, Recommendations, and Critical Reflection
**Author: [4th Member Name]**

---

## 4.1 Technical Evaluation

### Model Performance Metrics

The final K-Means model (K = 5) was evaluated using two complementary metrics:

**Inertia (Within-Cluster Sum of Squares):**  
Inertia measures the total squared distance between each data point and its assigned cluster centroid. Lower inertia indicates tighter, more compact clusters. The Elbow Method demonstrated that K = 5 achieves a strong reduction in inertia while avoiding overfitting through excessive cluster granularity.

**Silhouette Score:**  
The silhouette score ranges from −1 to +1. A score close to +1 indicates that data points are well-matched to their own cluster and poorly matched to neighbouring clusters. Our model achieved a silhouette score above 0.5, which Kaufman and Rousseeuw (1990) describe as indicative of a reasonable cluster structure. This provides statistical justification for the five-cluster solution.

### Limitations of K-Means

Despite its effectiveness here, K-Means carries several important limitations that must be acknowledged:

1. **Assumes spherical clusters:** K-Means minimises Euclidean distance, implicitly assuming that clusters are roughly circular. Non-spherical or irregularly shaped clusters would be poorly handled — DBSCAN would be more appropriate in such cases.
2. **Sensitivity to initialisation:** Although `k-means++` initialisation (Arthur and Vassilvitskii, 2007) was used to mitigate this, different random seeds can yield different cluster assignments.
3. **K must be specified in advance:** The algorithm requires K as an input, necessitating external validation (Elbow + Silhouette) rather than determining K automatically.
4. **Outlier sensitivity:** K-Means centroid positions can be pulled by extreme values, though this was not a significant concern here given the absence of major outliers identified in Stage 2.
5. **Static snapshot:** The model captures customer behaviour at a single point in time. Cluster membership may shift as customer income or spending patterns evolve.

---

**References:**  
Arthur, D. and Vassilvitskii, S. (2007) 'k-means++: The advantages of careful seeding', *Proceedings of the 18th Annual ACM-SIAM Symposium on Discrete Algorithms*, pp. 1027–1035.

Kaufman, L. and Rousseeuw, P.J. (1990) *Finding Groups in Data: An Introduction to Cluster Analysis*. New York: Wiley.

In [ ]:
# ─────────────────────────────────────────────
# Final Evaluation Metrics Summary
# ─────────────────────────────────────────────
final_sil = silhouette_score(X_scaled, df['Cluster'])
final_inertia = kmeans.inertia_

print('=' * 45)
print('      FINAL MODEL EVALUATION SUMMARY')
print('=' * 45)
print(f'  Algorithm        : K-Means (k-means++)')
print(f'  Number of Clusters (K) : {OPTIMAL_K}')
print(f'  Inertia          : {final_inertia:.2f}')
print(f'  Silhouette Score : {final_sil:.4f}')
print(f'  Dataset Size     : {len(df)} customers')
print('=' * 45)
print()
print('Cluster Sizes:')
print(df['Cluster'].value_counts().sort_index())

## 4.2 Business Evaluation and Strategic Recommendations

### Business Value

The segmentation model delivers concrete business value by enabling the organisation to replace undifferentiated mass marketing with a targeted, data-driven approach. The key value proposition is **efficiency**: marketing spend directed at the Premium Loyalists (Cluster 1) and Impulsive Spenders (Cluster 3) is likely to yield significantly higher conversion rates than campaigns distributed uniformly across all customers.

Research by McKinsey & Company (2021) indicates that personalisation at scale can deliver five to eight times the return on marketing investment. Even a conservative 2x improvement in campaign conversion rates would substantially offset the cost of the analytical infrastructure required to implement this model.

### Limitations and Risks

- **Dataset size:** At 200 records, the model is trained on a small sample. Production deployment would require validation against a larger, representative customer population before clusters can be considered stable.
- **Feature poverty:** The absence of product category data, purchase frequency, and recency information limits the depth of behavioural insight. An RFM (Recency, Frequency, Monetary) extension would substantially enrich the model.
- **Cluster drift:** Customer behaviour evolves over time. The model should be re-trained quarterly to ensure cluster assignments remain accurate.
- **Causal limitation:** Clustering identifies *correlation* in customer behaviour, not *causation*. Knowing that a customer has a high spending score does not explain *why* — qualitative research (surveys, interviews) should complement the quantitative segmentation.

### Actionable Recommendations

1. **Deploy personalised campaign streams:** Implement separate email, social media, and in-store promotion tracks for each of the five customer segments, with messaging tailored to each group's income-spending profile.
2. **Integrate with CRM system:** Assign each customer a cluster label within the existing CRM platform to enable real-time personalisation at every touchpoint.
3. **Enrich the dataset:** Collect purchase history, product preferences, and visit frequency to build a more granular second-generation segmentation model.
4. **Establish a measurement framework:** Define key performance indicators (KPIs) per segment — e.g., campaign click-through rate, average transaction value, and retention rate — and measure them before and after segment-targeted campaigns to quantify ROI.
5. **Conduct A/B testing:** Before full rollout, pilot targeted campaigns on a 20% sample of each cluster and compare performance against a control group to validate the model's business impact.

## 4.3 Ethical, Security, and Privacy Evaluation

### Ethics and Fairness

**Potential for Algorithmic Bias:**  
The dataset includes Gender as an attribute. While Gender was not used as a direct clustering feature, it is present in the dataset and correlates with other features. Post-hoc analysis of our cluster profiles reveals a gender imbalance in certain clusters (see Stage 3 profile summary). If marketing campaigns are designed purely on the basis of cluster membership — without awareness of this imbalance — the result may be systematically different experiences for male versus female customers, even without intent to discriminate. This constitutes a form of indirect or *proxy discrimination*, which is a recognised concern in algorithmic decision-making (Barocas and Selbst, 2016).

**Mitigation:**  
Cluster profiles should be reviewed through a fairness lens before campaign deployment. Marketing teams should ensure that clusters are not used to restrict access to promotions based on inferred demographic characteristics. Disaggregated cluster analysis by gender (and age group) should be conducted to detect and correct for inadvertent discriminatory patterns.

**Targeting Vulnerable Groups:**  
Cluster 3 (Impulsive Spenders) exhibits low income combined with high spending — a behavioural profile that may indicate financial vulnerability. Aggressive marketing tactics targeted at this group (e.g., promoting credit facilities or high-cost items) could cause real financial harm and may conflict with consumer protection principles embedded in UK consumer law (Competition and Markets Authority, 2021). Ethical marketing practice requires restraint in targeting this segment with offers that could encourage over-spending.

---

### Privacy

**Data Privacy Considerations:**  
Although the Mall Customer dataset is publicly available and does not contain directly identifiable information (e.g., names or addresses), privacy risks are not absent. The combination of Age, Annual Income, and Spending Score can constitute a quasi-identifier — a set of attributes that, when combined with external data sources, could enable re-identification of specific individuals (Narayanan and Shmatikoff, 2008). This is known as the *linkage attack* and is a well-documented risk in seemingly anonymised datasets.

**GDPR Implications:**  
In a real-world deployment, profiling customers for marketing purposes constitutes personal data processing under the UK General Data Protection Regulation (UK GDPR). Article 22 of UK GDPR restricts automated decision-making that produces significant effects on individuals without their explicit consent or another lawful basis (Information Commissioner's Office, 2023). Any deployment of this segmentation model in production would require:
- A documented lawful basis for processing (e.g., legitimate interests or consent).
- Transparent communication to customers about how their data is used for profiling.
- The provision of a right to opt out of automated profiling.

---

### Security

**Data Storage and Access Control:**  
If this system were deployed in a production environment, the customer dataset — which contains sensitive financial attributes — would need to be stored with appropriate encryption at rest (e.g., AES-256) and in transit (TLS 1.3). Role-based access control (RBAC) should restrict data access to authorised personnel only, with full audit logging of all access events.

**Model Security — Inference Attacks:**  
Deployed machine learning models are vulnerable to *model inversion attacks*, whereby an adversary queries the model to infer sensitive properties of the training data (Fredrikson et al., 2015). In a customer segmentation context, this could allow an attacker to infer income or spending patterns of specific customers. Mitigation strategies include rate-limiting model queries, adding differential privacy noise to model outputs, and never exposing raw training data through the API.

**Supply Chain Risk:**  
The analysis relies on third-party Python libraries (scikit-learn, pandas). Malicious packages or compromised dependencies represent a software supply chain risk. All dependencies should be version-pinned and sourced from trusted repositories, with periodic security audits.

---

### References

Barocas, S. and Selbst, A.D. (2016) 'Big data's disparate impact', *California Law Review*, 104(3), pp. 671–732.

Competition and Markets Authority (2021) *Consumer protection law compliance: A guide for businesses*. London: CMA.

Fredrikson, M., Jha, S. and Ristenpart, T. (2015) 'Model inversion attacks that exploit confidence information and basic countermeasures', *Proceedings of the 22nd ACM SIGSAC Conference on Computer and Communications Security*, pp. 1322–1333.

Information Commissioner's Office (2023) *Guide to the UK General Data Protection Regulation (UK GDPR)*. Available at: https://ico.org.uk/for-organisations/uk-gdpr-guidance-and-resources/ (Accessed: 10 March 2026).

Narayanan, A. and Shmatikoff, V. (2008) 'Robust de-anonymization of large sparse datasets', *Proceedings of the 2008 IEEE Symposium on Security and Privacy*, pp. 111–125.

---
## 4.4 Conclusion

This project has successfully demonstrated the end-to-end business intelligence and data mining pipeline applied to a real-world customer segmentation problem. Beginning with a clearly defined business question — how to group customers for targeted marketing — the team applied K-Means clustering to the Mall Customer dataset, identifying five statistically robust and business-interpretable customer segments.

Each segment was translated into an actionable marketing persona with specific strategic recommendations, moving beyond technical outputs to deliver genuine business intelligence. The evaluation demonstrated a strong silhouette score, validating the five-cluster structure, while the critical reflection identified important limitations relating to dataset size, temporal statics, and feature richness.

The ethical evaluation highlighted meaningful risks — including proxy discrimination, targeting of financially vulnerable customers, and GDPR compliance obligations — and proposed concrete mitigation strategies for each. The security analysis identified key risks for production deployment, including model inversion attacks and supply chain vulnerabilities.

In summary, the segmentation model provides a strong analytical foundation for differentiated marketing strategy, with clear pathways for enhancement through richer data collection, A/B testing, and CRM integration. The organisation that implements these recommendations stands to improve marketing ROI, enhance customer experience, and build a more sustainable, data-informed commercial strategy.

---
*End of Report*